# Анализ данных о самоубийствах (1985-2016)

Полный конвейер предобработки и анализа данных с различными визуализациями.

**Источник данных:** [Kaggle - Suicide Rates Overview 1985 to 2016](https://www.kaggle.com/datasets/russellyates88/suicide-rates-overview-1985-to-2016)

## 1. Установка и импорт библиотек

In [ ]:
# Установка необходимых библиотек
!pip install pandas plotly numpy pycountry seaborn matplotlib scipy scikit-learn -q
print("✓ Все библиотеки установлены")

In [3]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pycountry
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')

# Настройка стиля
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Все библиотеки загружены успешно")

✓ Все библиотеки загружены успешно


## 2. Загрузка исходных данных

In [4]:
# Загружаем исходные данные
print("⏳ Загрузка данных...")
df_raw = pd.read_csv('master.csv')

print(f"✓ Загружено {len(df_raw)} строк")
print(f"\nФорма данных: {df_raw.shape}")
print(f"\nПервые 5 строк:")
display(df_raw.head())
print(f"\nИнформация о данных:")
display(df_raw.info())
print(f"\nПропущенные значения:")
display(df_raw.isnull().sum())

⏳ Загрузка данных...
✓ Загружено 27820 строк

Форма данных: (27820, 12)

Первые 5 строк:


,country,year,sex,age,suicides_no,population,suicides/100k pop,country-year,HDI for year,gdp_for_year ($),gdp_per_capita ($),generation
0,Albania,1987,male,15-24 years,21,312900,6.71,Albania1987,NaN,"2,156,624,900",796,Generation X
1,Albania,1987,male,35-54 years,16,308000,5.19,Albania1987,NaN,"2,156,624,900",796,Silent
2,Albania,1987,female,15-24 years,14,289700,4.83,Albania1987,NaN,"2,156,624,900",796,Generation X
3,Albania,1987,male,75+ years,1,21800,4.59,Albania1987,NaN,"2,156,624,900",796,G.I. Generation
4,Albania,1987,male,25-34 years,9,274300,3.28,Albania1987,NaN,"2,156,624,900",796,Boomers



Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27820 entries, 0 to 27819
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             27820 non-null  object 
 1   year                27820 non-null  int64  
 2   sex                 27820 non-null  object 
 3   age                 27820 non-null  object 
 4   suicides_no         27820 non-null  int64  
 5   population          27820 non-null  int64  
 6   suicides/100k pop   27820 non-null  float64
 7   country-year        27820 non-null  object 
 8   HDI for year        8364 non-null   float64
 9    gdp_for_year ($)   27820 non-null  object 
 10  gdp_per_capita ($)  27820 non-null  int64  
 11  generation          27820 non-null  object 
dtypes: float64(2), int64(4), object(6)
memory usage: 2.5+ MB


None


Пропущенные значения:


country                   0
year                      0
sex                       0
age                       0
suicides_no               0
population                0
suicides/100k pop         0
country-year              0
HDI for year          19456
 gdp_for_year ($)         0
gdp_per_capita ($)        0
generation                0
dtype: int64

## 3. Предобработка данных

In [5]:
def get_iso_alpha3_code(country_name: str) -> str:
    """
    Преобразует название страны в ISO-3 код.
    """
    country_mapping = {
        'Russian Federation': 'RUS', 'United States': 'USA', 'United Kingdom': 'GBR',
        'South Korea': 'KOR', 'Bahamas': 'BHS', 'Bosnia and Herzegovina': 'BIH',
        'Czech Republic': 'CZE', 'Côte d\'Ivoire': 'CIV', 'Dominican Republic': 'DOM',
        'El Salvador': 'SLV', 'Kyrgyzstan': 'KGZ', 'Mauritius': 'MUS',
        'Moldova': 'MDA', 'Saint Lucia': 'LCA', 'Saint Vincent and Grenadines': 'VCT',
        'Trinidad and Tobago': 'TTO', 'Turkmenistan': 'TKM', 'United Arab Emirates': 'ARE',
        'Uzbekistan': 'UZB', 'Venezuela': 'VEN', 'Serbia': 'SRB',
        'Montenegro': 'MNE', 'Tajikistan': 'TJK', 'Taiwan': 'TWN',
        'Georgia': 'GEO', 'Guyana': 'GUY', 'Suriname': 'SUR', 'Kazakhstan': 'KAZ'
    }
    
    if country_name in country_mapping:
        return country_mapping[country_name]
    
    try:
        country = pycountry.countries.search_fuzzy(country_name)
        if country:
            return country[0].alpha_3
    except (AttributeError, LookupError):
        pass
    
    return None


print("⏳ Начало предобработки...")

# Копируем данные
df = df_raw.copy()

# 1. Обработка пропущенных значений
print("\n1️⃣ Обработка пропущенных значений")
initial_nulls = df.isnull().sum().sum()

# Заполняем HDI среднее значение по странам
df['HDI for year'] = df.groupby('country')['HDI for year'].transform(
    lambda x: x.fillna(x.mean())
)

# Удаляем строки где все еще есть NaN в важных колонках
df = df.dropna(subset=['suicides_no', 'population', 'year', 'country'])

final_nulls = df.isnull().sum().sum()
print(f"  ✓ Пропущенных значений уменьшилось с {initial_nulls} до {final_nulls}")

# 2. Преобразование типов данных
print("\n2️⃣ Преобразование типов данных")
df['year'] = df['year'].astype('int32')
df['suicides_no'] = df['suicides_no'].astype('int32')
df['population'] = df['population'].astype('int32')
df['HDI for year'] = pd.to_numeric(df['HDI for year'], errors='coerce')
print("  ✓ Типы данных преобразованы")

# 3. Добавление производных признаков
print("\n3️⃣ Добавление производных признаков")

# Коэффициент самоубийств на 100k
df['suicide_rate'] = (df['suicides_no'] / df['population'] * 100000).round(2)

# Возрастная группа (числовой индекс для сортировки)
age_order = {
    '5-14 years': 0, '15-24 years': 1, '25-34 years': 2, '35-54 years': 3,
    '55-74 years': 4, '75+ years': 5
}
df['age_group_order'] = df['age'].map(age_order)

# ISO-3 коды стран
print("  Преобразование названий стран в ISO-3 коды...")
df['iso_alpha'] = df['country'].apply(get_iso_alpha3_code)

# Удаляем неизвестные страны
initial_rows = len(df)
df = df[df['iso_alpha'].notna()]
print(f"  ✓ Удалено {initial_rows - len(df)} записей для неизвестных стран")

# 4. Выявление и обработка выбросов
print("\n4️⃣ Выявление выбросов")

# Вычисляем медиану и IQR
Q1 = df['suicide_rate'].quantile(0.25)
Q3 = df['suicide_rate'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = len(df[(df['suicide_rate'] < lower_bound) | (df['suicide_rate'] > upper_bound)])
print(f"  ✓ Выявлено {outliers} выбросов (не удаляем, используем для анализа)")
print(f"    Диапазон нормальных значений: [{lower_bound:.2f}, {upper_bound:.2f}]")

# 5. Стандартизация текстовых данных
print("\n5️⃣ Стандартизация текстовых данных")
df['sex'] = df['sex'].str.lower().str.strip()
df['country'] = df['country'].str.strip()
df['age'] = df['age'].str.strip()
print(f"  ✓ Текстовые данные стандартизированы")

print(f"\n✓ Предобработка завершена!")
print(f"\nФинальная форма данных: {df.shape}")
print(f"Период: {df['year'].min()} - {df['year'].max()}")
print(f"Количество стран: {df['country'].nunique()}")
print(f"Количество возрастных групп: {df['age'].nunique()}")

⏳ Начало предобработки...

1️⃣ Обработка пропущенных значений
  ✓ Пропущенных значений уменьшилось с 19456 до 1486

2️⃣ Преобразование типов данных
  ✓ Типы данных преобразованы

3️⃣ Добавление производных признаков
  Преобразование названий стран в ISO-3 коды...
  ✓ Удалено 96 записей для неизвестных стран

4️⃣ Выявление выбросов
  ✓ Выявлено 2032 выбросов (не удаляем, используем для анализа)
    Диапазон нормальных значений: [-22.71, 40.30]

5️⃣ Стандартизация текстовых данных
  ✓ Текстовые данные стандартизированы

✓ Предобработка завершена!

Финальная форма данных: (27724, 15)
Период: 1985 - 2016
Количество стран: 99
Количество возрастных групп: 6


## 4. Исследовательский анализ данных (EDA)

In [8]:
# Базовая статистика
print("📊 БАЗОВАЯ СТАТИСТИКА")
print("\nСтатистика коэффициента самоубийств:")
print(df['suicide_rate'].describe().round(2))

print("\n\nРаспределение по полам:")
print(df['sex'].value_counts())

print("\n\nРаспределение по возрастным группам:")
age_dist = df.groupby('age')['suicide_rate'].agg(['count', 'mean', 'std']).round(2)
age_dist = age_dist.reindex([k for k, v in sorted(age_order.items(), key=lambda x: x[1])])
display(age_dist)

📊 БАЗОВАЯ СТАТИСТИКА

Статистика коэффициента самоубийств:
count    27724.00
mean        12.85
std         18.98
min          0.00
25%          0.92
50%          6.03
75%         16.67
max        224.97
Name: suicide_rate, dtype: float64


Распределение по полам:
sex
male      13862
female    13862
Name: count, dtype: int64


Распределение по возрастным группам:


,count,mean,std
age,,,
5-14 years,4594,0.62,1.02
15-24 years,4626,8.97,9.59
25-34 years,4626,12.22,14.12
35-54 years,4626,14.99,17.72
55-74 years,4626,16.20,18.26
75+ years,4626,24.00,30.27


## 5. Визуализация 1: Тренд во времени

In [9]:
# Средний уровень самоубийств по годам
yearly_trend = df.groupby('year').agg({
    'suicide_rate': ['mean', 'median', 'std'],
    'suicides_no': 'sum'
}).round(2)

yearly_trend.columns = ['Mean', 'Median', 'Std', 'Total']
yearly_trend = yearly_trend.reset_index()

fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=yearly_trend['year'],
    y=yearly_trend['Mean'],
    name='Среднее',
    line=dict(color='#d62728', width=3),
    mode='lines+markers'
))

fig1.add_trace(go.Scatter(
    x=yearly_trend['year'],
    y=yearly_trend['Median'],
    name='Медиана',
    line=dict(color='#1f77b4', width=2, dash='dash'),
    mode='lines+markers'
))

fig1.update_layout(
    title='Тренд уровня самоубийств во времени (1985-2016)',
    xaxis_title='Год',
    yaxis_title='Самоубийства на 100k населения',
    height=500,
    hovermode='x unified',
    template='plotly_white'
)

fig1.show()
print("✓ График сохранен")

✓ График сохранен


## 6. Визуализация 2: Распределение по полам

In [11]:
# Сравнение по полам
sex_data = df.groupby(['sex', 'year'])['suicide_rate'].mean().reset_index()

fig2 = px.line(
    sex_data,
    x='year',
    y='suicide_rate',
    color='sex',
    title='Уровень самоубийств по полам',
    labels={'suicide_rate': 'Самоубийства на 100k', 'year': 'Год', 'sex': 'Пол'},
    markers=True
)

fig2.update_layout(
    height=500,
    hovermode='x unified',
    template='plotly_white'
)

fig2.show()

# Статистика по полам
print("\nСредний уровень самоубийств по полам:")
sex_stats = df.groupby('sex')['suicide_rate'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
display(sex_stats)


Средний уровень самоубийств по полам:


,mean,median,std,min,max
sex,,,,,
female,5.40,3.18,7.36,0.0,133.42
male,20.29,13.62,23.57,0.0,224.97


## 7. Визуализация 3: Распределение по возрастным группам

In [13]:
# Боксплот по возрастам
fig3 = go.Figure()

ages_sorted = [k for k, v in sorted(age_order.items(), key=lambda x: x[1])]

for age in ages_sorted:
    age_data = df[df['age'] == age]['suicide_rate']
    fig3.add_trace(go.Box(
        y=age_data,
        name=age,
        boxmean='sd'
    ))

fig3.update_layout(
    title='Распределение уровня самоубийств по возрастным группам',
    yaxis_title='Самоубийства на 100k',
    height=500,
    template='plotly_white'
)

fig3.show()

# Средний уровень по возрастам
print("\nСредний уровень самоубийств по возрастным группам:")
age_stats = df.groupby('age')['suicide_rate'].agg(['mean', 'median', 'count']).round(2)
age_stats = age_stats.reindex(ages_sorted)
display(age_stats)


Средний уровень самоубийств по возрастным группам:


,mean,median,count
age,,,
5-14 years,0.62,0.32,4594
15-24 years,8.97,5.93,4626
25-34 years,12.22,7.29,4626
35-54 years,14.99,9.12,4626
55-74 years,16.20,10.48,4626
75+ years,24.00,12.50,4626


## 8. Визуализация 4: Топ-20 стран по среднему уровню самоубийств

In [19]:
# Топ-20 стран
top_countries = df.groupby('country')['suicide_rate'].mean().sort_values(ascending=True).tail(20)

fig4 = go.Figure(go.Bar(
    y=top_countries.index,
    x=top_countries.values,
    orientation='h',
    marker=dict(
        color=top_countries.values,
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title='Уровень')
    )
))

fig4.update_layout(
    title='Топ-20 стран с наибольшим уровнем самоубийств (среднее за 1985-2016)',
    xaxis_title='Самоубийства на 100k',
    height=600,
    template='plotly_white'
)

fig4.show()

print("\nТоп-10 стран:")
top_countries.tail(10).round(2)


Топ-10 стран:


country
Ukraine               26.58
Estonia               27.28
Slovenia              27.83
Latvia                29.26
Kazakhstan            30.51
Belarus               31.08
Hungary               32.76
Russian Federation    34.89
Sri Lanka             35.30
Lithuania             40.42
Name: suicide_rate, dtype: float64

## 9. Визуализация 5: Тепловая карта корреляции

In [21]:
# Подготовка данных для корреляции
corr_data = df[['year', 'suicide_rate', 'HDI for year']].dropna().corr()

fig5 = go.Figure(data=go.Heatmap(
    z=corr_data.values,
    x=corr_data.columns,
    y=corr_data.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_data.values.round(3),
    texttemplate='%{text}',
    textfont={"size": 12}
))

fig5.update_layout(
    title='Матрица корреляции',
    height=500,
    template='plotly_white'
)

fig5.show()

print("\nМатрица корреляции:")
corr_data.round(3)


Матрица корреляции:


,year,suicide_rate,HDI for year
year,1.000,-0.046,0.140
suicide_rate,-0.046,1.000,0.116
HDI for year,0.140,0.116,1.000


## 10. Визуализация 6: Сравнение по полам и возрасту

In [22]:
# Построение тепловой карты
heatmap_data = df.pivot_table(
    values='suicide_rate',
    index='age',
    columns='sex',
    aggfunc='mean'
)

# Сортировка по возрастным группам
heatmap_data = heatmap_data.reindex([k for k, v in sorted(age_order.items(), key=lambda x: x[1])])

fig6 = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='YlOrRd',
    text=heatmap_data.values.round(1),
    texttemplate='%{text}',
    textfont={"size": 11}
))

fig6.update_layout(
    title='Средний уровень самоубийств: Пол × Возраст',
    xaxis_title='Пол',
    yaxis_title='Возрастная группа',
    height=500,
    template='plotly_white'
)

fig6.show()

## 11. Визуализация 7: Интерактивная карта мира (2016)

In [36]:
def create_interactive_map(data: pd.DataFrame, initial_year: int = 2016) -> go.Figure:
    """
    Создает интерактивную карту мира с фреймами для каждого года.

    Args:
        data: DataFrame с данными о самоубийствах
        initial_year: начальный год для отображения

    Returns:
        Фигура Plotly для интерактивной визуализации
    """

    # Получаем список всех лет в возрастающем порядке
    years = sorted(data['year'].unique())

    # Создаем фреймы для каждого года
    frames = []

    for year in years:
        year_data = data[data['year'] == year]

        frame = go.Frame(
            data=[
                go.Choropleth(
                    locations=year_data['country'],
                    z=year_data['suicide_rate'],
                    locationmode='country names',
                    colorscale='Reds',
                    text=year_data['country'],
                    hovertemplate='<b>%{text}</b><br>' +
                                  'Уровень самоубийств: %{z:.2f} на 100k<extra></extra>',
                    colorbar=dict(
                        title='Самоубийства<br>на 100k<br>населения',
                        thickness=15,
                        len=0.7,
                        x=1.02
                    ),
                    zmin=data['suicide_rate'].min(),
                    zmax=data['suicide_rate'].max()
                )
            ],
            name=str(year),
            layout=go.Layout(title_text=f'Уровень самоубийств по странам: {year}')
        )
        frames.append(frame)

    # Данные для начального года
    initial_data = data[data['year'] == initial_year]

    # Создаем главную фигуру
    fig = go.Figure(
        data=[
            go.Choropleth(
                locations=initial_data['country'],
                z=initial_data['suicide_rate'],
                locationmode='country names',
                colorscale='Reds',
                text=initial_data['country'],
                hovertemplate='<b>%{text}</b><br>' +
                              'Уровень самоубийств: %{z:.2f} на 100k<extra></extra>',
                colorbar=dict(
                    title='Самоубийства<br>на 100k<br>населения',
                    thickness=15,
                    len=0.7,
                    x=1.02
                ),
                zmin=data['suicide_rate'].min(),
                zmax=data['suicide_rate'].max()
            )
        ],
        frames=frames
    )

    # Создаем слайдер
    sliders = [
        {
            'active': years.index(initial_year),
            'yanchor': 'top',
            'y': 0,
            'xanchor': 'left',
            'x': 0.1,
            'len': 0.8,
            'transition': {'duration': 300},
            'pad': {'b': 10, 't': 50},
            'currentvalue': {
                'prefix': 'Год: ',
                'visible': True,
                'xanchor': 'center',
                'font': {'size': 16, 'color': '#555'}
            },
            'steps': [
                {
                    'args': [
                        [str(year)],
                        {
                            'frame': {'duration': 300, 'redraw': True},
                            'mode': 'immediate',
                            'transition': {'duration': 300}
                        }
                    ],
                    'label': str(year),
                    'method': 'animate'
                }
                for year in years
            ]
        }
    ]

    # Кнопки управления (Play/Pause)
    updatemenus = [
        {
            'type': 'buttons',
            'showactive': False,
            'y': 0,
            'x': 0,
            'xanchor': 'left',
            'yanchor': 'top',
            'pad': {'t': 70, 'r': 10},
            'buttons': [
                {
                    'label': '▶ Проиграть',
                    'method': 'animate',
                    'args': [None, {
                        'frame': {'duration': 500, 'redraw': True},
                        'fromcurrent': True,
                        'transition': {'duration': 300, 'easing': 'quadratic-in-out'}
                    }]
                },
                {
                    'label': '⏸ Пауза',
                    'method': 'animate',
                    'args': [[None], {
                        'frame': {'duration': 0, 'redraw': True},
                        'mode': 'immediate',
                        'transition': {'duration': 0}
                    }]
                }
            ]
        }
    ]

    # Обновляем макет
    fig.update_layout(
        title={
            'text': f'Уровень самоубийств по странам: {initial_year}',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 20, 'color': '#333'}
        },
        geo=dict(
            projection_type='natural earth',
            bgcolor='rgba(240, 240, 240, 0.5)',
            coastlinecolor='#555',
            showocean=True,
            oceancolor='#e5f3ff'
        ),
        height=700,
        margin=dict(l=0, r=150, t=100, b=100),
        font=dict(family='Arial, sans-serif', size=12),
        sliders=sliders,
        updatemenus=updatemenus,
        hovermode='closest'
    )

    return fig


# Создаем интерактивную карту
print("⏳ Создание интерактивной карты...")
fig = create_interactive_map(df, initial_year=2016)
print("✓ Карта создана успешно")
fig.show()

⏳ Создание интерактивной карты...
✓ Карта создана успешно


## 12. Визуализация 8: Отношение мужчин к женщинам

In [24]:
# Вычисляем отношение
male_female_ratio = df[df['sex'] == 'male'].groupby('country')['suicide_rate'].mean() / \
                   df[df['sex'] == 'female'].groupby('country')['suicide_rate'].mean()

male_female_ratio = male_female_ratio.sort_values(ascending=False).head(15)

fig8 = go.Figure(go.Bar(
    x=male_female_ratio.values,
    y=male_female_ratio.index,
    orientation='h',
    marker=dict(
        color=male_female_ratio.values,
        colorscale='Blues',
        showscale=True
    )
))

fig8.update_layout(
    title='Отношение самоубийств мужчин к женщинам (среднее за период)',
    xaxis_title='Отношение (М/Ж)',
    height=500,
    template='plotly_white'
)

fig8.show()

# print(f"\nСредний коэффициент отношения М/Ж: {male_female_ratio.mean():.2f}")
# print(f"Мужчины совершают в среднем в {male_female_ratio.mean():.1f} раз больше самоубийств, чем женщины")


Средний коэффициент отношения М/Ж: inf
Мужчины совершают в среднем в inf раз больше самоубийств, чем женщины


## 13. Визуализация 9: Диаграмма размаха (Scatter plot)

In [26]:
# Корреляция между ВВП на душу и уровнем самоубийств
data_scatter = df[['country', 'year', 'suicide_rate', 'gdp_per_capita ($)']].dropna()
data_scatter = data_scatter.groupby('country').agg({
    'suicide_rate': 'mean',
    'gdp_per_capita ($)': 'mean'
}).reset_index()

fig9 = px.scatter(
    data_scatter,
    x='gdp_per_capita ($)',
    y='suicide_rate',
    hover_name='country',
    title='Взаимосвязь: ВВП на душу населения vs Уровень самоубийств',
    labels={'gdp_per_capita ($)': 'ВВП на душу (USD)', 'suicide_rate': 'Самоубийства на 100k'},
    size='suicide_rate',
    color='suicide_rate',
    #colorscale='Reds'
)

fig9.update_layout(
    height=500,
    hovermode='closest',
    template='plotly_white'
)

fig9.show()

# Коэффициент корреляции
corr_gdp = data_scatter['gdp_per_capita ($)'].corr(data_scatter['suicide_rate'])
print(f"\nКоэффициент корреляции (ВВП на душу vs Самоубийства): {corr_gdp:.3f}")


Коэффициент корреляции (ВВП на душу vs Самоубийства): 0.036


## 14. Визуализация 10: Изменение за период по странам

In [28]:
# Вычисляем изменение
country_change = []

for country in df['country'].unique():
    country_data = df[df['country'] == country].sort_values('year')
    if len(country_data) > 5:
        first_rate = country_data.iloc[0]['suicide_rate']
        last_rate = country_data.iloc[-1]['suicide_rate']
        change = last_rate - first_rate
        pct_change = (change / first_rate * 100) if first_rate != 0 else 0
        
        country_change.append({
            'country': country,
            'change': change,
            'pct_change': pct_change,
            'first_rate': first_rate,
            'last_rate': last_rate
        })

change_df = pd.DataFrame(country_change).sort_values('change')

# Топ 10 увеличений и снижений
top_increase = change_df.tail(10)
top_decrease = change_df.head(10)

fig10 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Топ-10: Увеличение', 'Топ-10: Снижение'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}]]
)

fig10.add_trace(
    go.Bar(y=top_increase['country'], x=top_increase['change'], orientation='h', name='Увеличение', marker_color='#d62728'),
    row=1, col=1
)

fig10.add_trace(
    go.Bar(y=top_decrease['country'], x=top_decrease['change'], orientation='h', name='Снижение', marker_color='#2ca02c'),
    row=1, col=2
)

fig10.update_xaxes(title_text='Изменение (на 100k)', row=1, col=1)
fig10.update_xaxes(title_text='Изменение (на 100k)', row=1, col=2)

fig10.update_layout(
    title='Изменение уровня самоубийств по странам (1985-2016)',
    height=500,
    showlegend=False,
    template='plotly_white'
)

fig10.show()

print("\nТоп-10 стран с НАИБОЛЬШИМ УВЕЛИЧЕНИЕМ:")
display(top_increase[['country', 'change', 'pct_change']])
print("\nТоп-10 стран с НАИБОЛЬШИМ СНИЖЕНИЕМ:")
display(top_decrease[['country', 'change', 'pct_change']])


Топ-10 стран с НАИБОЛЬШИМ УВЕЛИЧЕНИЕМ:


,country,change,pct_change
49,Kuwait,-2.84,-100.000000
67,Philippines,-1.97,-87.168142
24,Cyprus,-1.57,-100.000000
14,Bosnia and Herzegovina,-0.55,-100.000000
54,Maldives,0.00,0.000000
1,Antigua and Barbuda,0.00,0.000000
27,Dominica,0.00,0.000000
59,Montenegro,0.00,0.000000
75,Saint Kitts and Nevis,0.00,0.000000
71,Qatar,0.00,0.000000



Топ-10 стран с НАИБОЛЬШИМ СНИЖЕНИЕМ:


,country,change,pct_change
4,Aruba,-224.97,-100.000000
40,Hungary,-174.30,-98.524674
81,Singapore,-144.16,-99.661251
52,Lithuania,-141.30,-97.549189
16,Bulgaria,-131.17,-100.000000
86,Sri Lanka,-124.54,-99.456956
30,Estonia,-124.06,-98.884106
23,Cuba,-120.30,-100.000000
33,France,-120.18,-99.742717
79,Serbia,-118.79,-99.748090


## 15. Итоговая статистика и выводы

In [29]:
print("="*70)
print("ИТОГОВАЯ СТАТИСТИКА")
print("="*70)

print(f"\n📊 Общие данные:")
print(f"  • Период анализа: {df['year'].min()} - {df['year'].max()} ({df['year'].max() - df['year'].min() + 1} лет)")
print(f"  • Количество стран: {df['country'].nunique()}")
print(f"  • Количество записей: {len(df):,}")
print(f"  • Всего самоубийств в датасете: {df['suicides_no'].sum():,}")

print(f"\n🔢 Статистика по уровню самоубийств (на 100k):")
print(f"  • Среднее: {df['suicide_rate'].mean():.2f}")
print(f"  • Медиана: {df['suicide_rate'].median():.2f}")
print(f"  • Стандартное отклонение: {df['suicide_rate'].std():.2f}")
print(f"  • Минимум: {df['suicide_rate'].min():.2f}")
print(f"  • Максимум: {df['suicide_rate'].max():.2f}")

print(f"\n👥 Разница по полам:")
male_rate = df[df['sex'] == 'male']['suicide_rate'].mean()
female_rate = df[df['sex'] == 'female']['suicide_rate'].mean()
print(f"  • Мужчины: {male_rate:.2f} на 100k")
print(f"  • Женщины: {female_rate:.2f} на 100k")
print(f"  • Отношение М/Ж: {male_rate/female_rate:.2f}x")

print(f"\n📈 Тренд во времени:")
first_year_rate = df[df['year'] == df['year'].min()]['suicide_rate'].mean()
last_year_rate = df[df['year'] == df['year'].max()]['suicide_rate'].mean()
trend_change = last_year_rate - first_year_rate
trend_pct = (trend_change / first_year_rate * 100)
print(f"  • {df['year'].min()}: {first_year_rate:.2f} на 100k")
print(f"  • {df['year'].max()}: {last_year_rate:.2f} на 100k")
print(f"  • Изменение: {trend_change:+.2f} ({trend_pct:+.1f}%)")

print(f"\n👴 Распределение по возрастам:")
for age in [k for k, v in sorted(age_order.items(), key=lambda x: x[1])]:
    age_rate = df[df['age'] == age]['suicide_rate'].mean()
    print(f"  • {age}: {age_rate:.2f} на 100k")

print("\n" + "="*70)

ИТОГОВАЯ СТАТИСТИКА

📊 Общие данные:
  • Период анализа: 1985 - 2016 (32 лет)
  • Количество стран: 99
  • Количество записей: 27,724
  • Всего самоубийств в датасете: 6,738,262

🔢 Статистика по уровню самоубийств (на 100k):
  • Среднее: 12.85
  • Медиана: 6.03
  • Стандартное отклонение: 18.98
  • Минимум: 0.00
  • Максимум: 224.97

👥 Разница по полам:
  • Мужчины: 20.29 на 100k
  • Женщины: 5.40 на 100k
  • Отношение М/Ж: 3.76x

📈 Тренд во времени:
  • 1985: 11.83 на 100k
  • 2016: 13.42 на 100k
  • Изменение: +1.60 (+13.5%)

👴 Распределение по возрастам:
  • 5-14 years: 0.62 на 100k
  • 15-24 years: 8.97 на 100k
  • 25-34 years: 12.22 на 100k
  • 35-54 years: 14.99 на 100k
  • 55-74 years: 16.20 на 100k
  • 75+ years: 24.00 на 100k



## 16. Сохранение обработанных данных

In [30]:
# Сохраняем обработанные данные
df.to_csv('processed_suicide_data.csv', index=False)
print("✓ Обработанные данные сохранены в 'processed_suicide_data.csv'")

# Сохраняем агрегированные данные
agg_data = df.groupby('country').agg({
    'suicide_rate': ['mean', 'min', 'max', 'std'],
    'suicides_no': 'sum',
    'iso_alpha': 'first'
}).round(2)
agg_data.to_csv('aggregated_suicide_data.csv')
print("✓ Агрегированные данные сохранены в 'aggregated_suicide_data.csv'")

✓ Обработанные данные сохранены в 'processed_suicide_data.csv'
✓ Агрегированные данные сохранены в 'aggregated_suicide_data.csv'
